# Shifu OCR — Colab Training v3
**Automatic:** Mounts Google Drive, pulls engine + training data, trains, saves model back to Drive.

**Setup (once):** Upload these to your Google Drive root:
- `shifu_engine.zip` (21 KB)
- `training_data.zip` (257 MB)

Then just **Run All**.

In [ ]:
# Step 1: Install + Mount Drive
!pip install numpy Pillow scipy scikit-image -q

from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# Step 2: Auto-extract engine + training data from Drive
import os, zipfile

DRIVE = '/content/drive/MyDrive'

# Extract engine code
engine_zip = os.path.join(DRIVE, 'shifu_engine.zip')
if os.path.exists(engine_zip):
    zipfile.ZipFile(engine_zip).extractall('.')
    print(f'Engine extracted from Drive')
else:
    print(f'ERROR: Put shifu_engine.zip in Google Drive root')

# Extract training data
data_zip = os.path.join(DRIVE, 'training_data.zip')
if os.path.exists(data_zip):
    print('Extracting training data (this takes a minute)...')
    zipfile.ZipFile(data_zip).extractall('training_data')
    print('Training data extracted')
else:
    print(f'ERROR: Put training_data.zip in Google Drive root')

TRAINING_DIR = 'training_data'
if os.path.exists('training_data/training_data'):
    TRAINING_DIR = 'training_data/training_data'
img_dir = os.path.join(TRAINING_DIR, 'images')
n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
print(f'Found {n_images} training images')

# Verify engine
import numpy as np
from shifu_ocr.engine import ShifuOCR
ocr = ShifuOCR()
test_img = np.full((100,100), 255, dtype=np.uint8)
test_img[30:70, 40:60] = 0
fv = ocr._extract_unified_features(test_img)
print(f'Feature dimensions: {len(fv)}')

In [ ]:
# Step 3: Train
import time, multiprocessing as mp
from PIL import Image
from shifu_ocr.engine import ShifuOCR, Landscape

ALLOWED = set('abcdefghijklmnopqrstuvwxyz0123456789')

def process_image(args):
    img_path, label = args
    try:
        from shifu_ocr.engine import ShifuOCR
        import numpy as np
        _ocr = ShifuOCR()
        img = Image.open(img_path)
        if img.mode != 'L': img = img.convert('L')
        arr = np.array(img)
        img.close()
        segments = _ocr.segment_characters(arr, min_char_width=2)
        if not segments: return []
        clean = [c.lower() for c in label if c.lower() in ALLOWED]
        if not clean: return []
        ratio = len(segments) / max(len(clean), 1)
        if ratio < 0.8 or ratio > 1.2: return []
        n = min(len(segments), len(clean))
        return [(clean[i], _ocr._extract_unified_features(segments[i]['image']).tolist()) for i in range(n)]
    except:
        return []

entries = []
with open(os.path.join(TRAINING_DIR, 'train_list.txt'), 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or '\t' not in line: continue
        p, l = line.split('\t', 1)
        fp = os.path.join(TRAINING_DIR, p)
        if os.path.exists(fp): entries.append((fp, l))
print(f'{len(entries)} images, {mp.cpu_count()} cores')

landscapes = {}
total = 0
start = time.time()
BS = 2000

for bi in range(0, len(entries), BS):
    batch = entries[bi:bi+BS]
    with mp.Pool(mp.cpu_count()) as pool:
        results = pool.map(process_image, batch)
    for cl in results:
        for lbl, fv in cl:
            if lbl not in landscapes: landscapes[lbl] = Landscape(lbl)
            landscapes[lbl].absorb(np.array(fv))
            total += 1
    el = time.time() - start
    bn = bi//BS+1
    tb = (len(entries)+BS-1)//BS
    print(f'  Batch {bn}/{tb}: {total} chars, {total/max(el,1):.0f}/sec, {el:.0f}s')

print(f'\nDone: {total} chars, {len(landscapes)} landscapes, {(time.time()-start)/60:.1f} min')

In [ ]:
# Step 4: Save to Drive + local download
import json
model = {
    'version': '2.0.0',
    'total_predictions': 0,
    'total_correct': 0,
    'landscapes': {k: v.to_dict() for k, v in landscapes.items()},
}

# Save locally
with open('trained_model.json', 'w') as f:
    json.dump(model, f)

# Also save to Drive for easy access
import shutil
shutil.copy('trained_model.json', os.path.join(DRIVE, 'trained_model.json'))

size_mb = os.path.getsize('trained_model.json') / 1024 / 1024
dim = len(list(landscapes.values())[0].mean)
print(f'Model: {size_mb:.1f} MB, {dim} features, {len(landscapes)} landscapes')
print(f'Saved to Google Drive: {DRIVE}/trained_model.json')

In [ ]:
# Step 5: Validation
val_entries = []
vp = os.path.join(TRAINING_DIR, 'val_list.txt')
if os.path.exists(vp):
    with open(vp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line: continue
            p, l = line.split('\t', 1)
            fp = os.path.join(TRAINING_DIR, p)
            if os.path.exists(fp): val_entries.append((fp, l))

print(f'Validating on {min(len(val_entries), 300)} images...')
test_ocr = ShifuOCR()
test_ocr.landscapes = landscapes
correct = tested = 0
for ip, lb in val_entries[:300]:
    try:
        img = Image.open(ip).convert('L')
        r = test_ocr.read_line(np.array(img))
        img.close()
        pred = r['text'].lower().replace(' ','')
        truth = ''.join(c.lower() for c in lb if c.lower() in ALLOWED)
        for a, b in zip(pred, truth):
            tested += 1
            if a == b: correct += 1
    except: pass
print(f'Character accuracy: {correct}/{tested} ({100*correct/max(tested,1):.1f}%)')

In [ ]:
# Step 6: Download locally too
from google.colab import files
files.download('trained_model.json')
print('Also available at Google Drive: MyDrive/trained_model.json')